# MossFormer2 speech-separation comparison

This runs the same five hand-labeled overlapping exchanges used for the SepFormer tests. It processes one short window at a time to bound memory use, preserves both blind separator outputs, and applies ECAPA only after separation. Enable Internet and a GPU, and attach the stage-checkpoints dataset containing `full-video.wav`.


In [ ]:
from pathlib import Path
import base64, json, os, shutil, subprocess, sys

REVISION = "mossformer2-five-exchange-v1"
BASE = Path("/kaggle/working")
WORK = BASE/"mossformer2-comparison"
OUTPUT = BASE/"mossformer2-results"
VENV = BASE/"diarization-venv"
PYTHON = str(VENV/"bin"/"python")
for directory in (WORK, OUTPUT): directory.mkdir(parents=True, exist_ok=True)
print("Revision:", REVISION)
print("Output:", OUTPUT)


## Install the isolated runtime


In [ ]:
EMBEDDED_FILES = {'review_overlap_extraction.py': '"""Supplemental target-speaker extraction for baseline overlap '
                                 'intervals.\n'
                                 '\n'
                                 'The baseline file is read-only. Results are review candidates '
                                 'and never replace\n'
                                 'baseline text, timing, evidence, confidence, or speaker '
                                 'identity.\n'
                                 '"""\n'
                                 '\n'
                                 'import argparse\n'
                                 'from collections import Counter\n'
                                 'from difflib import SequenceMatcher\n'
                                 'import json\n'
                                 'from pathlib import Path\n'
                                 'import re\n'
                                 'import subprocess\n'
                                 '\n'
                                 'import numpy as np\n'
                                 '\n'
                                 '\n'
                                 'def select_overlap_segments(baseline, policy=None):\n'
                                 '    """Select the additive union of baseline and strong '
                                 'supplemental review rows."""\n'
                                 '    selected_by_index = {}\n'
                                 '    for index, segment in enumerate(baseline.get("segments", '
                                 '[])):\n'
                                 '        overlap = next(\n'
                                 '            (\n'
                                 '                item\n'
                                 '                for item in segment.get("evidence", [])\n'
                                 '                if item.get("source") == "overlapping_speakers"\n'
                                 '                and item.get("details", '
                                 '{}).get("target_and_non_target", False)\n'
                                 '            ),\n'
                                 '            None,\n'
                                 '        )\n'
                                 '        if overlap is not None:\n'
                                 '            selected_by_index[index] = (index, segment, '
                                 'overlap)\n'
                                 '    if policy is not None:\n'
                                 '        segments = baseline.get("segments", [])\n'
                                 '        for row in policy.get("segments", []):\n'
                                 '            if row.get("tier") != "separation_review":\n'
                                 '                continue\n'
                                 '            index = int(row["baseline_index"])\n'
                                 '            if not 0 <= index < len(segments):\n'
                                 '                raise ValueError(f"Policy baseline index is out '
                                 'of range: {index}")\n'
                                 '            segment = segments[index]\n'
                                 '            if (abs(float(row["start"]) - '
                                 'float(segment["start"])) > 0.02\n'
                                 '                    or abs(float(row["end"]) - '
                                 'float(segment["end"])) > 0.02):\n'
                                 '                raise ValueError(f"Policy timing does not match '
                                 'baseline index {index}")\n'
                                 '            if index not in selected_by_index:\n'
                                 '                selected_by_index[index] = (index, segment, {\n'
                                 '                    "source": "supplemental_overlap_policy",\n'
                                 '                    "details": {\n'
                                 '                        "tier": row["tier"],\n'
                                 '                        "reasons": row.get("reasons", []),\n'
                                 '                        "sortformer_overlap_fraction": row.get(\n'
                                 '                            "sortformer_overlap_fraction", '
                                 '0.0),\n'
                                 '                        "diaper_overlap_fraction": row.get(\n'
                                 '                            "diaper_overlap_fraction", 0.0),\n'
                                 '                    },\n'
                                 '                })\n'
                                 '    return [selected_by_index[index] for index in '
                                 'sorted(selected_by_index)]\n'
                                 '\n'
                                 '\n'
                                 'def classify_extraction(original_similarity, '
                                 'extracted_similarity, energy_retention,\n'
                                 '                        transcript):\n'
                                 '    """Triage only; every result remains review-required."""\n'
                                 '    similarity_gain = extracted_similarity - '
                                 'original_similarity\n'
                                 '    if energy_retention < 0.10:\n'
                                 '        return "likely_suppressed_residual"\n'
                                 '    if transcript.strip() and energy_retention >= 0.10 and '
                                 'similarity_gain >= 0.10:\n'
                                 '        return "candidate_target_speech"\n'
                                 '    return "unresolved"\n'
                                 '\n'
                                 '\n'
                                 'def unit(vector):\n'
                                 '    vector = np.asarray(vector, dtype=np.float32).reshape(-1)\n'
                                 '    return vector / max(float(np.linalg.norm(vector)), 1e-9)\n'
                                 '\n'
                                 '\n'
                                 'def rms(wave):\n'
                                 '    return float(np.sqrt(np.mean(np.asarray(wave, '
                                 'dtype=np.float32) ** 2)))\n'
                                 '\n'
                                 '\n'
                                 'def stereo_metrics(wave):\n'
                                 '    """Measure whether stereo contains information beyond '
                                 'duplicated mono."""\n'
                                 '    wave = np.asarray(wave, dtype=np.float32)\n'
                                 '    if wave.ndim != 2 or wave.shape[1] != 2 or len(wave) < 2:\n'
                                 '        return {"available": False, "distinct": False}\n'
                                 '    left, right = wave[:, 0], wave[:, 1]\n'
                                 '    middle, side = (left + right) / 2, (left - right) / 2\n'
                                 '    correlation = float(np.corrcoef(left, right)[0, 1])\n'
                                 '    side_to_middle_db = float(20 * np.log10(\n'
                                 '        (rms(side) + 1e-12) / (rms(middle) + 1e-12)\n'
                                 '    ))\n'
                                 '    # Lossy encoders can make duplicated channels differ by tiny '
                                 'amounts. Analyze\n'
                                 '    # channels only when the difference is large enough to carry '
                                 'real content.\n'
                                 '    distinct = bool(np.isfinite(correlation) and correlation < '
                                 '0.98\n'
                                 '                    and side_to_middle_db >= -25.0)\n'
                                 '    return {\n'
                                 '        "available": True,\n'
                                 '        "distinct": distinct,\n'
                                 '        "correlation": correlation,\n'
                                 '        "side_to_middle_db": side_to_middle_db,\n'
                                 '        "left_to_right_level_db": float(20 * np.log10(\n'
                                 '            (rms(left) + 1e-12) / (rms(right) + 1e-12)\n'
                                 '        )),\n'
                                 '    }\n'
                                 '\n'
                                 '\n'
                                 'def stereo_signals(wave):\n'
                                 '    wave = np.asarray(wave, dtype=np.float32)\n'
                                 '    left, right = wave[:, 0], wave[:, 1]\n'
                                 '    return {\n'
                                 '        "left": left,\n'
                                 '        "right": right,\n'
                                 '        "middle": (left + right) / 2,\n'
                                 '        "difference": (left - right) / 2,\n'
                                 '    }\n'
                                 '\n'
                                 '\n'
                                 'def corroborated_novel_words(transcriptions, baseline_text, '
                                 'minimum_views=2):\n'
                                 '    """Return inserted words decoded in independent views.\n'
                                 '\n'
                                 '    Replacement hypotheses such as sue/see or city/scene are '
                                 'transcription\n'
                                 '    disagreements, not recovered concurrent speech, and are '
                                 'intentionally\n'
                                 '    excluded here.\n'
                                 '    """\n'
                                 '    tokens = lambda text: re.findall(r"[a-z0-9\']+", '
                                 'str(text).casefold())\n'
                                 '    baseline_words = tokens(baseline_text)\n'
                                 '    support = Counter()\n'
                                 '    views = {}\n'
                                 '    for name, text in transcriptions.items():\n'
                                 '        candidate_words = tokens(text)\n'
                                 '        inserted = set()\n'
                                 '        for tag, _, _, candidate_start, candidate_end in '
                                 'SequenceMatcher(\n'
                                 '                None, baseline_words, '
                                 'candidate_words).get_opcodes():\n'
                                 '            if tag == "insert":\n'
                                 '                '
                                 'inserted.update(candidate_words[candidate_start:candidate_end])\n'
                                 '        for word in inserted:\n'
                                 '            support[word] += 1\n'
                                 '            views.setdefault(word, []).append(name)\n'
                                 '    return [\n'
                                 '        {"word": word, "support": support[word], "views": '
                                 'sorted(views[word])}\n'
                                 '        for word in sorted(support)\n'
                                 '        if support[word] >= minimum_views\n'
                                 '    ]\n'
                                 '\n'
                                 '\n'
                                 'def transcribe(model, wave):\n'
                                 '    segments, _ = model.transcribe(\n'
                                 '        wave,\n'
                                 '        vad_filter=False,\n'
                                 '        condition_on_previous_text=False,\n'
                                 '        beam_size=5,\n'
                                 '    )\n'
                                 '    rows = list(segments)\n'
                                 '    return {\n'
                                 '        "text": " ".join(row.text.strip() for row in rows if '
                                 'row.text.strip()),\n'
                                 '        "average_log_probability": (\n'
                                 '            float(np.mean([row.avg_logprob for row in rows])) if '
                                 'rows else None\n'
                                 '        ),\n'
                                 '        "maximum_no_speech_probability": (\n'
                                 '            float(max(row.no_speech_prob for row in rows)) if '
                                 'rows else None\n'
                                 '        ),\n'
                                 '    }\n'
                                 '\n'
                                 '\n'
                                 'def main():\n'
                                 '    parser = argparse.ArgumentParser()\n'
                                 '    parser.add_argument("--video", type=Path, required=True)\n'
                                 '    parser.add_argument("--baseline", type=Path, required=True)\n'
                                 '    parser.add_argument("--enrollment", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--voice-priors", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--output-dir", type=Path, '
                                 'required=True)\n'
                                 '    parser.add_argument("--device", default="cuda")\n'
                                 '    parser.add_argument("--whisper-model", default="large-v2")\n'
                                 '    parser.add_argument("--wesep-model-dir", type=Path)\n'
                                 '    parser.add_argument("--maximum-segments", type=int)\n'
                                 '    parser.add_argument("--selection-policy", type=Path)\n'
                                 '    args = parser.parse_args()\n'
                                 '\n'
                                 '    import soundfile as sf\n'
                                 '    import torch\n'
                                 '    import torchaudio\n'
                                 '    import wesep\n'
                                 '    from faster_whisper import WhisperModel\n'
                                 '    from speechbrain.inference.speaker import '
                                 'SpeakerRecognition\n'
                                 '\n'
                                 '    args.output_dir.mkdir(parents=True, exist_ok=True)\n'
                                 '    baseline_bytes = args.baseline.read_bytes()\n'
                                 '    baseline = json.loads(baseline_bytes)\n'
                                 '    policy = (json.loads(args.selection_policy.read_text())\n'
                                 '              if args.selection_policy else None)\n'
                                 '    selected = select_overlap_segments(baseline, policy)\n'
                                 '    if args.maximum_segments is not None:\n'
                                 '        selected = selected[:args.maximum_segments]\n'
                                 '\n'
                                 '    source_stereo = args.output_dir / "source-stereo-16khz.wav"\n'
                                 '    subprocess.run(\n'
                                 '        [\n'
                                 '            "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", '
                                 '"error", "-y",\n'
                                 '            "-i", str(args.video), "-vn", "-ac", "2", "-ar", '
                                 '"16000",\n'
                                 '            str(source_stereo),\n'
                                 '        ],\n'
                                 '        check=True,\n'
                                 '    )\n'
                                 '    full_stereo, sample_rate = sf.read(\n'
                                 '        source_stereo, dtype="float32", always_2d=True\n'
                                 '    )\n'
                                 '    if sample_rate != 16000:\n'
                                 '        raise RuntimeError(f"Unexpected extracted sample rate: '
                                 '{sample_rate}")\n'
                                 '    full_wave = full_stereo.mean(axis=1)\n'
                                 '    source_audio = args.output_dir / "source-16khz.wav"\n'
                                 '    sf.write(source_audio, full_wave, sample_rate)\n'
                                 '\n'
                                 '    extractor = (\n'
                                 '        wesep.load_model_local(str(args.wesep_model_dir))\n'
                                 '        if args.wesep_model_dir\n'
                                 '        else wesep.load_model("english")\n'
                                 '    )\n'
                                 '    extractor.set_device(args.device)\n'
                                 '    extractor.set_vad(True)\n'
                                 '    # Preserve attenuation so residual noise can be rejected.\n'
                                 '    extractor.set_output_norm(False)\n'
                                 '\n'
                                 '    target_rows = np.load(args.voice_priors)\n'
                                 '    target = unit(np.mean(np.stack([unit(row) for row in '
                                 'target_rows]), axis=0))\n'
                                 '    speaker_dir = '
                                 'Path("pretrained_models/spkrec-ecapa-voxceleb")\n'
                                 '    speaker = SpeakerRecognition.from_hparams(\n'
                                 '        source=str(speaker_dir),\n'
                                 '        savedir=str(speaker_dir),\n'
                                 '        run_opts={"device": args.device},\n'
                                 '    )\n'
                                 '    whisper_device = "cuda" if args.device.startswith("cuda") '
                                 'else "cpu"\n'
                                 '    whisper = WhisperModel(\n'
                                 '        args.whisper_model,\n'
                                 '        device=whisper_device,\n'
                                 '        compute_type="float16" if whisper_device == "cuda" else '
                                 '"int8",\n'
                                 '    )\n'
                                 '\n'
                                 '    extracted_dir = args.output_dir / "audio"\n'
                                 '    extracted_dir.mkdir(exist_ok=True)\n'
                                 '    results = []\n'
                                 '    for number, (baseline_index, segment, overlap) in '
                                 'enumerate(selected, 1):\n'
                                 '        start, end = float(segment["start"]), '
                                 'float(segment["end"])\n'
                                 '        left, right = max(0, round(start * sample_rate)), min(\n'
                                 '            len(full_wave), round(end * sample_rate)\n'
                                 '        )\n'
                                 '        original = full_wave[left:right]\n'
                                 '        original_stereo = full_stereo[left:right]\n'
                                 '        if len(original) < 1:\n'
                                 '            continue\n'
                                 '        original_path = args.output_dir / '
                                 'f"current-{baseline_index:04d}.wav"\n'
                                 '        sf.write(original_path, original, sample_rate)\n'
                                 '        extracted_tensor = extractor.extract_speech(\n'
                                 '            str(original_path), str(args.enrollment)\n'
                                 '        )\n'
                                 '        if extracted_tensor is None:\n'
                                 '            results.append({\n'
                                 '                "baseline_index": baseline_index,\n'
                                 '                "start": start,\n'
                                 '                "end": end,\n'
                                 '                "baseline_text": segment.get("text", ""),\n'
                                 '                "status": "extractor_returned_no_speech",\n'
                                 '                "review_required": True,\n'
                                 '            })\n'
                                 '            continue\n'
                                 '        extracted = extracted_tensor[0].detach().cpu().numpy()\n'
                                 '        extracted_path = extracted_dir / '
                                 'f"overlap-{baseline_index:04d}-target.wav"\n'
                                 '        sf.write(extracted_path, extracted, sample_rate)\n'
                                 '\n'
                                 '        def similarity(wave):\n'
                                 '            tensor = torch.from_numpy(np.asarray(wave, '
                                 'dtype=np.float32)).unsqueeze(0)\n'
                                 '            embedding = unit(\n'
                                 '                '
                                 'speaker.encode_batch(tensor).flatten().detach().cpu().numpy()\n'
                                 '            )\n'
                                 '            return float(np.dot(target, embedding))\n'
                                 '\n'
                                 '        original_similarity = similarity(original)\n'
                                 '        extracted_similarity = similarity(extracted)\n'
                                 '        retention = rms(extracted) / max(rms(original), 1e-9)\n'
                                 '        original_asr = transcribe(whisper, original)\n'
                                 '        extracted_asr = transcribe(whisper, extracted)\n'
                                 '        status = classify_extraction(\n'
                                 '            original_similarity,\n'
                                 '            extracted_similarity,\n'
                                 '            retention,\n'
                                 '            extracted_asr["text"],\n'
                                 '        )\n'
                                 '        channel_metrics = stereo_metrics(original_stereo)\n'
                                 '        stereo_review = {\n'
                                 '            "analyzed": False,\n'
                                 '            "metrics": channel_metrics,\n'
                                 '            "status": "channels_not_distinct",\n'
                                 '            "review_required": True,\n'
                                 '        }\n'
                                 '        if channel_metrics.get("distinct", False):\n'
                                 '            channel_audio = stereo_signals(original_stereo)\n'
                                 '            channel_results = {}\n'
                                 '            for name, wave in channel_audio.items():\n'
                                 '                channel_results[name] = {\n'
                                 '                    "target_similarity": similarity(wave),\n'
                                 '                    "rms": rms(wave),\n'
                                 '                    "transcription": transcribe(whisper, wave),\n'
                                 '                }\n'
                                 '            transcriptions = {\n'
                                 '                name: value["transcription"]["text"]\n'
                                 '                for name, value in channel_results.items()\n'
                                 '            }\n'
                                 '            novel = corroborated_novel_words(\n'
                                 '                transcriptions, segment.get("text", "")\n'
                                 '            )\n'
                                 '            stereo_review = {\n'
                                 '                "analyzed": True,\n'
                                 '                "metrics": channel_metrics,\n'
                                 '                "signals": channel_results,\n'
                                 '                "corroborated_novel_words": novel,\n'
                                 '                "status": ("corroborated_words_for_review" if '
                                 'novel\n'
                                 '                           else '
                                 '"distinct_channels_no_corroborated_new_words"),\n'
                                 '                "speaker": "Uncertain",\n'
                                 '                "review_required": True,\n'
                                 '                "note": (\n'
                                 '                    "Channel decoding is supplemental evidence. '
                                 'Words require "\n'
                                 '                    "speaker review and are never inserted into '
                                 'the baseline."\n'
                                 '                ),\n'
                                 '            }\n'
                                 '        results.append({\n'
                                 '            "baseline_index": baseline_index,\n'
                                 '            "start": start,\n'
                                 '            "end": end,\n'
                                 '            "baseline_text": segment.get("text", ""),\n'
                                 '            "baseline_speaker": segment.get("final_speaker"),\n'
                                 '            "baseline_confidence": '
                                 'segment.get("final_confidence"),\n'
                                 '            "overlap": overlap.get("details", {}),\n'
                                 '            "original": {\n'
                                 '                "target_similarity": original_similarity,\n'
                                 '                "rms": rms(original),\n'
                                 '                "transcription": original_asr,\n'
                                 '            },\n'
                                 '            "extracted": {\n'
                                 '                "target_similarity": extracted_similarity,\n'
                                 '                "similarity_gain": extracted_similarity - '
                                 'original_similarity,\n'
                                 '                "rms": rms(extracted),\n'
                                 '                "energy_retention": retention,\n'
                                 '                "transcription": extracted_asr,\n'
                                 '                "audio": '
                                 'str(extracted_path.relative_to(args.output_dir)),\n'
                                 '            },\n'
                                 '            "stereo": stereo_review,\n'
                                 '            "status": status,\n'
                                 '            "review_required": True,\n'
                                 '            "baseline_modified": False,\n'
                                 '        })\n'
                                 '        print(\n'
                                 '            f"Overlap {number}/{len(selected)} at {start:.2f}s: '
                                 '{status}; "\n'
                                 '            f"retention={retention:.3f}; '
                                 'similarity={original_similarity:.3f}"\n'
                                 '            f"->{extracted_similarity:.3f}; '
                                 '{extracted_asr[\'text\']}",\n'
                                 '            flush=True,\n'
                                 '        )\n'
                                 '\n'
                                 '    counts = {}\n'
                                 '    stereo_counts = {}\n'
                                 '    for row in results:\n'
                                 '        counts[row["status"]] = counts.get(row["status"], 0) + '
                                 '1\n'
                                 '        stereo_status = row.get("stereo", {}).get("status", '
                                 '"not_available")\n'
                                 '        stereo_counts[stereo_status] = '
                                 'stereo_counts.get(stereo_status, 0) + 1\n'
                                 '    report = {\n'
                                 '        "review_required": True,\n'
                                 '        "baseline_modified": False,\n'
                                 '        "selection": (\n'
                                 '            "additive baseline overlap plus strong supplemental '
                                 'policy"\n'
                                 '            if policy is not None\n'
                                 '            else "target/non-target diarization overlap '
                                 'evidence"\n'
                                 '        ),\n'
                                 '        "selection_policy": str(args.selection_policy) if '
                                 'args.selection_policy else None,\n'
                                 '        "thresholds_are_provisional": True,\n'
                                 '        "triage_thresholds": {\n'
                                 '            "suppressed_below_energy_retention": 0.10,\n'
                                 '            "candidate_minimum_energy_retention": 0.10,\n'
                                 '            "candidate_minimum_similarity_gain": 0.10,\n'
                                 '            "stereo_maximum_channel_correlation": 0.98,\n'
                                 '            "stereo_minimum_side_to_middle_db": -25.0,\n'
                                 '            "stereo_novel_word_minimum_views": 2,\n'
                                 '        },\n'
                                 '        "summary": {"selected": len(selected), "completed": '
                                 'len(results),\n'
                                 '                    "status": counts, "stereo_status": '
                                 'stereo_counts},\n'
                                 '        "segments": results,\n'
                                 '    }\n'
                                 '    (args.output_dir / '
                                 '"report.json").write_text(json.dumps(report, indent=2) + "\\n")\n'
                                 '    lines = [\n'
                                 '        f"[{row[\'start\']:.2f}-{row[\'end\']:.2f}] '
                                 '{row[\'status\']}: "\n'
                                 '        f"{row.get(\'extracted\', {}).get(\'transcription\', '
                                 '{}).get(\'text\', \'\')}; "\n'
                                 '        f"stereo={row.get(\'stereo\', {}).get(\'status\', '
                                 '\'not_available\')}; "\n'
                                 '        f"novel={\',\'.join(item[\'word\'] for item in '
                                 "row.get('stereo', {}).get('corroborated_novel_words', "
                                 '[]))}"\n'
                                 '        for row in results\n'
                                 '    ]\n'
                                 '    (args.output_dir / '
                                 '"review.txt").write_text("\\n".join(lines) + "\\n")\n'
                                 '    if args.baseline.read_bytes() != baseline_bytes:\n'
                                 '        raise RuntimeError("Baseline changed during supplemental '
                                 'extraction review")\n'
                                 '\n'
                                 '\n'
                                 'if __name__ == "__main__":\n'
                                 '    main()\n',
 'run_contextual_wesep_experiment.py': '#!/usr/bin/env python3\n'
                                       '"""Run contextual target-speaker extraction on a fixed '
                                       'labeled benchmark."""\n'
                                       '\n'
                                       'import argparse\n'
                                       'import json\n'
                                       'from pathlib import Path\n'
                                       'import re\n'
                                       'import subprocess\n'
                                       'import sys\n'
                                       'import types\n'
                                       '\n'
                                       'import numpy as np\n'
                                       '\n'
                                       'from review_overlap_extraction import rms, transcribe, '
                                       'unit\n'
                                       '\n'
                                       '\n'
                                       'def crop_context_output(wave, sample_rate, window_start, '
                                       'interval_start, interval_end):\n'
                                       '    left = round((interval_start - window_start) * '
                                       'sample_rate)\n'
                                       '    right = round((interval_end - window_start) * '
                                       'sample_rate)\n'
                                       '    return np.asarray(wave, dtype=np.float32)[max(0, '
                                       'left):max(0, right)]\n'
                                       '\n'
                                       '\n'
                                       'def token_f1(reference, hypothesis):\n'
                                       '    words=lambda x: '
                                       're.findall(r"[a-z0-9\']+",str(x).casefold())\n'
                                       '    ref, hyp=words(reference),words(hypothesis)\n'
                                       '    if not ref or not hyp:return 0.0\n'
                                       '    remaining=list(ref);hits=0\n'
                                       '    for word in hyp:\n'
                                       '        if word in '
                                       'remaining:hits+=1;remaining.remove(word)\n'
                                       '    precision=hits/len(hyp);recall=hits/len(ref)\n'
                                       '    return 2*precision*recall/max(precision+recall,1e-9)\n'
                                       '\n'
                                       '\n'
                                       'def main():\n'
                                       '    '
                                       'p=argparse.ArgumentParser();p.add_argument("--video",type=Path,required=True);p.add_argument("--labels",type=Path,required=True);p.add_argument("--enrollment",type=Path,required=True);p.add_argument("--voice-priors",type=Path,required=True);p.add_argument("--output-dir",type=Path,required=True);p.add_argument("--context",type=float,action="append",default=[]);p.add_argument("--device",default="cpu");p.add_argument("--whisper-model",default="large-v2");p.add_argument("--wesep-model-dir",type=Path)\n'
                                       '    a=p.parse_args();contexts=a.context or '
                                       '[3.0,5.0];a.output_dir.mkdir(parents=True,exist_ok=True)\n'
                                       '    import soundfile as sf\n'
                                       '    import torch\n'
                                       '    # WeSep imports an unused UMAP diarization dependency. '
                                       'Avoid initializing its\n'
                                       '    # Numba cache in portable/local environments; '
                                       'extraction does not use UMAP.\n'
                                       '    if "umap" not in sys.modules:\n'
                                       '        '
                                       'module=types.ModuleType("umap");module.UMAP=object;sys.modules["umap"]=module\n'
                                       '    import wesep\n'
                                       '    from faster_whisper import WhisperModel\n'
                                       '    from speechbrain.inference.speaker import '
                                       'SpeakerRecognition\n'
                                       '    labels=json.loads(a.labels.read_text())["labels"]\n'
                                       '    source=a.output_dir/"source-16khz.wav"\n'
                                       '    '
                                       'subprocess.run(["ffmpeg","-nostdin","-hide_banner","-loglevel","error","-y","-i",str(a.video),"-vn","-ac","1","-ar","16000",str(source)],check=True)\n'
                                       '    full,sr=sf.read(source,dtype="float32");assert '
                                       'sr==16000 and full.ndim==1\n'
                                       '    '
                                       'extractor=(wesep.load_model_local(str(a.wesep_model_dir)) '
                                       'if a.wesep_model_dir else '
                                       'wesep.load_model("english"));extractor.set_device(a.device);extractor.set_vad(True);extractor.set_output_norm(False)\n'
                                       '    '
                                       'priors=np.load(a.voice_priors);target=unit(np.mean(np.stack([unit(x) '
                                       'for x in priors]),axis=0))\n'
                                       '    '
                                       'spkdir=Path("pretrained_models/spkrec-ecapa-voxceleb")\n'
                                       '    '
                                       'speaker=SpeakerRecognition.from_hparams(source=str(spkdir),savedir=str(spkdir),run_opts={"device":a.device})\n'
                                       '    whisper=WhisperModel(a.whisper_model,device="cuda" if '
                                       'a.device.startswith("cuda") else '
                                       '"cpu",compute_type="float16" if '
                                       'a.device.startswith("cuda") else "int8")\n'
                                       '    '
                                       'audio=a.output_dir/"audio";audio.mkdir(exist_ok=True);rows=[]\n'
                                       '    def similarity(w):\n'
                                       '        '
                                       'tensor=torch.from_numpy(np.asarray(w,dtype=np.float32)).unsqueeze(0)\n'
                                       '        '
                                       'emb=unit(speaker.encode_batch(tensor).flatten().detach().cpu().numpy())\n'
                                       '        return float(np.dot(target,emb))\n'
                                       '    for label in labels:\n'
                                       '        '
                                       'start,end=float(label["start"]),float(label["end"])\n'
                                       '        for context in contexts:\n'
                                       '            '
                                       'ws=max(0.0,start-context);we=min(len(full)/sr,end+context)\n'
                                       '            '
                                       'window=full[round(ws*sr):round(we*sr)];stem=f"{label[\'exchange_id\']}-context-{context:g}s"\n'
                                       '            '
                                       'window_path=audio/f"{stem}-input.wav";sf.write(window_path,window,sr)\n'
                                       '            '
                                       'tensor=extractor.extract_speech(str(window_path),str(a.enrollment))\n'
                                       '            if tensor is None:\n'
                                       '                '
                                       'rows.append({"exchange_id":label["exchange_id"],"baseline_index":label["baseline_index"],"context_seconds":context,"status":"no_speech"});continue\n'
                                       '            separated=tensor[0].detach().cpu().numpy()\n'
                                       '            '
                                       'cropped=crop_context_output(separated,sr,ws,start,end)\n'
                                       '            expected=round((end-start)*sr)\n'
                                       '            if '
                                       'len(cropped)<expected:cropped=np.pad(cropped,(0,expected-len(cropped)))\n'
                                       '            cropped=cropped[:expected]\n'
                                       '            '
                                       'path=audio/f"{stem}-target.wav";sf.write(path,cropped,sr)\n'
                                       '            '
                                       'asr=transcribe(whisper,cropped);text=asr["text"]\n'
                                       '            '
                                       'row={"exchange_id":label["exchange_id"],"baseline_index":label["baseline_index"],"start":start,"end":end,"context_seconds":context,"status":"completed","output_length_samples":len(separated),"input_length_samples":len(window),"alignment_preserved":len(separated)==len(window),"audio":str(path.relative_to(a.output_dir)),"target_similarity":similarity(cropped),"energy_retention":rms(cropped)/max(rms(full[round(start*sr):round(end*sr)]),1e-9),"transcription":asr,"target_word_f1":token_f1(label.get("target_words",""),text),"other_word_f1":token_f1(label.get("other_words",""),text)}\n'
                                       '            rows.append(row);print(f"{stem}: '
                                       "sim={row['target_similarity']:.3f} "
                                       "target_f1={row['target_word_f1']:.2f} "
                                       "other_f1={row['other_word_f1']:.2f}: "
                                       '{text}",flush=True)\n'
                                       '    '
                                       'report={"schema_version":1,"method":"contextual_wesep","labels":str(a.labels.resolve()),"labels_used_for_extraction":False,"contexts":contexts,"results":rows}\n'
                                       '    '
                                       '(a.output_dir/"report.json").write_text(json.dumps(report,indent=2)+"\\n")\n'
                                       '\n'
                                       'if __name__=="__main__":main()\n',
 'run_mossformer2_separation_experiment.py': '#!/usr/bin/env python3\n'
                                             '"""Compare MossFormer2\'s two blind outputs on the '
                                             'fixed overlap benchmark."""\n'
                                             '\n'
                                             'import argparse\n'
                                             'import json\n'
                                             'import os\n'
                                             'import subprocess\n'
                                             'from pathlib import Path\n'
                                             '\n'
                                             'import numpy as np\n'
                                             '\n'
                                             'from review_overlap_extraction import transcribe, '
                                             'unit\n'
                                             'from run_contextual_wesep_experiment import '
                                             'token_f1\n'
                                             '\n'
                                             '\n'
                                             'def main():\n'
                                             '    parser = argparse.ArgumentParser()\n'
                                             '    source_group = '
                                             'parser.add_mutually_exclusive_group(required=True)\n'
                                             '    source_group.add_argument("--video", type=Path)\n'
                                             '    source_group.add_argument("--audio", type=Path)\n'
                                             '    parser.add_argument("--labels", type=Path, '
                                             'required=True)\n'
                                             '    parser.add_argument("--voice-priors", type=Path, '
                                             'required=True)\n'
                                             '    parser.add_argument("--output-dir", type=Path, '
                                             'required=True)\n'
                                             '    parser.add_argument("--context", type=float, '
                                             'default=3.0)\n'
                                             '    parser.add_argument("--device", default="cpu")\n'
                                             '    parser.add_argument("--whisper-model", '
                                             'default="large-v2")\n'
                                             '    parser.add_argument("--hf-home", type=Path)\n'
                                             '    args = parser.parse_args()\n'
                                             '\n'
                                             "    # ClearVoice's librosa import uses Numba "
                                             'caching. An explicit writable cache\n'
                                             '    # avoids failures when site-packages is '
                                             'read-only or lacks a source locator.\n'
                                             '    os.environ.setdefault("NUMBA_CACHE_DIR", '
                                             '"/tmp/numba-clearvoice")\n'
                                             '    if args.hf_home:\n'
                                             '        os.environ["HF_HOME"] = str(args.hf_home)\n'
                                             '\n'
                                             '    import soundfile as sf\n'
                                             '    import torch\n'
                                             '    from clearvoice import ClearVoice\n'
                                             '    from faster_whisper import WhisperModel\n'
                                             '    from speechbrain.inference.speaker import '
                                             'SpeakerRecognition\n'
                                             '\n'
                                             '    args.output_dir.mkdir(parents=True, '
                                             'exist_ok=True)\n'
                                             '    audio_dir = args.output_dir / "audio"\n'
                                             '    audio_dir.mkdir(exist_ok=True)\n'
                                             '    source = args.output_dir / "source-16khz.wav"\n'
                                             '    input_path = args.video or args.audio\n'
                                             '    command = [\n'
                                             '        "ffmpeg", "-nostdin", "-hide_banner", '
                                             '"-loglevel", "error", "-y",\n'
                                             '        "-i", str(input_path),\n'
                                             '    ]\n'
                                             '    if args.video:\n'
                                             '        command.append("-vn")\n'
                                             '    command.extend(["-ac", "1", "-ar", "16000", '
                                             'str(source)])\n'
                                             '    subprocess.run(command, check=True)\n'
                                             '    full, sample_rate = sf.read(source, '
                                             'dtype="float32")\n'
                                             '    if sample_rate != 16000:\n'
                                             '        raise RuntimeError(f"Unexpected sample rate: '
                                             '{sample_rate}")\n'
                                             '\n'
                                             '    labels = '
                                             'json.loads(args.labels.read_text())["labels"]\n'
                                             '    windows = []\n'
                                             '    for label in labels:\n'
                                             '        start, end = float(label["start"]), '
                                             'float(label["end"])\n'
                                             '        window_start = max(0.0, start - '
                                             'args.context)\n'
                                             '        window_end = min(len(full) / sample_rate, '
                                             'end + args.context)\n'
                                             '        wave = full[round(window_start * '
                                             'sample_rate):round(window_end * sample_rate)]\n'
                                             '        windows.append((label, window_start, '
                                             'window_end, wave))\n'
                                             '\n'
                                             '    separator = ClearVoice(\n'
                                             '        task="speech_separation", '
                                             'model_names=["MossFormer2_SS_16K"]\n'
                                             '    )\n'
                                             '\n'
                                             '    speaker_dir = Path(os.environ.get(\n'
                                             '        "SPEECHBRAIN_CACHE", '
                                             '"pretrained_models/spkrec-ecapa-voxceleb"\n'
                                             '    ))\n'
                                             '    speaker = SpeakerRecognition.from_hparams(\n'
                                             '        source="speechbrain/spkrec-ecapa-voxceleb", '
                                             'savedir=str(speaker_dir),\n'
                                             '        run_opts={"device": args.device},\n'
                                             '    )\n'
                                             '    whisper = WhisperModel(\n'
                                             '        args.whisper_model,\n'
                                             '        device="cuda" if '
                                             'args.device.startswith("cuda") else "cpu",\n'
                                             '        compute_type="float16" if '
                                             'args.device.startswith("cuda") else "int8",\n'
                                             '    )\n'
                                             '    priors = np.load(args.voice_priors)\n'
                                             '    target = unit(np.mean(np.stack([unit(row) for '
                                             'row in priors]), axis=0))\n'
                                             '\n'
                                             '    def similarity(wave):\n'
                                             '        tensor = torch.from_numpy(np.asarray(wave, '
                                             'dtype=np.float32)).unsqueeze(0)\n'
                                             '        embedding = unit(\n'
                                             '            '
                                             'speaker.encode_batch(tensor).flatten().detach().cpu().numpy()\n'
                                             '        )\n'
                                             '        return float(np.dot(target, embedding))\n'
                                             '\n'
                                             '    results = []\n'
                                             '    for item_index, (label, window_start, '
                                             'window_end, original) in enumerate(windows):\n'
                                             '        # Decode one short window at a time. '
                                             'MossFormer2 has a large activation\n'
                                             '        # footprint; batching all benchmark windows '
                                             'can exhaust a 16 GB laptop.\n'
                                             '        decoded = '
                                             'np.asarray(separator(original[None, :]))\n'
                                             '        if decoded.shape[:2] == (2, 1):\n'
                                             '            decoded = np.transpose(decoded, (1, 0, '
                                             '2))\n'
                                             '        if decoded.shape[:2] != (1, 2):\n'
                                             '            raise RuntimeError(f"Unexpected '
                                             'MossFormer2 output shape: {decoded.shape}")\n'
                                             '        start, end = float(label["start"]), '
                                             'float(label["end"])\n'
                                             '        left = round((start - window_start) * '
                                             'sample_rate)\n'
                                             '        right = round((end - window_start) * '
                                             'sample_rate)\n'
                                             '        streams = []\n'
                                             '        for stream_index in range(2):\n'
                                             '            whole = decoded[0, stream_index, '
                                             ':len(original)]\n'
                                             '            cropped = np.asarray(whole[left:right], '
                                             'dtype=np.float32)\n'
                                             '            path = audio_dir / '
                                             'f"{label[\'exchange_id\']}-stream-{stream_index + '
                                             '1}.wav"\n'
                                             '            sf.write(path, cropped, sample_rate)\n'
                                             '            asr = transcribe(whisper, cropped)\n'
                                             '            streams.append({\n'
                                             '                "stream": stream_index + 1,\n'
                                             '                "audio": '
                                             'str(path.relative_to(args.output_dir)),\n'
                                             '                "target_similarity": '
                                             'similarity(cropped),\n'
                                             '                "transcription": asr,\n'
                                             '                "target_word_f1": '
                                             'token_f1(label.get("target_words", ""), '
                                             'asr["text"]),\n'
                                             '                "other_word_f1": '
                                             'token_f1(label.get("other_words", ""), '
                                             'asr["text"]),\n'
                                             '            })\n'
                                             '        ranked = sorted(streams, key=lambda row: '
                                             'row["target_similarity"], reverse=True)\n'
                                             '        results.append({\n'
                                             '            "exchange_id": label["exchange_id"],\n'
                                             '            "baseline_index": '
                                             'label["baseline_index"],\n'
                                             '            "start": start,\n'
                                             '            "end": end,\n'
                                             '            "context_seconds": args.context,\n'
                                             '            "window_start": window_start,\n'
                                             '            "window_end": window_end,\n'
                                             '            "target_words": '
                                             'label.get("target_words", ""),\n'
                                             '            "other_words": label.get("other_words", '
                                             '""),\n'
                                             '            "streams": streams,\n'
                                             '            "similarity_selected_stream": '
                                             'ranked[0]["stream"],\n'
                                             '            "similarity_margin": '
                                             'ranked[0]["target_similarity"] - '
                                             'ranked[1]["target_similarity"],\n'
                                             '        })\n'
                                             '        print(\n'
                                             '            label["exchange_id"],\n'
                                             '            "; ".join(\n'
                                             '                f"s{row[\'stream\']} '
                                             'sim={row[\'target_similarity\']:.3f}: "\n'
                                             '                '
                                             'f"{row[\'transcription\'][\'text\']}" for row in '
                                             'streams\n'
                                             '            ),\n'
                                             '            flush=True,\n'
                                             '        )\n'
                                             '\n'
                                             '    report = {\n'
                                             '        "schema_version": 1,\n'
                                             '        "method": '
                                             '"blind_mossformer2_ss_16k_then_ecapa",\n'
                                             '        "labels_used_for_separation": False,\n'
                                             '        "context_seconds": args.context,\n'
                                             '        "results": results,\n'
                                             '    }\n'
                                             '    (args.output_dir / '
                                             '"report.json").write_text(json.dumps(report, '
                                             'indent=2) + "\\n")\n'
                                             '\n'
                                             '\n'
                                             'if __name__ == "__main__":\n'
                                             '    main()\n'}
for name, source in EMBEDDED_FILES.items():
    (WORK/name).write_text(source)
(WORK/"labels.json").write_text('{\n  "schema_version": 1,\n  "purpose": "fixed_ground_truth_for_separation_experiments",\n  "labels": [\n    {\n      "exchange_id": "01-0033",\n      "baseline_index": 33,\n      "start": 76.417,\n      "end": 77.778,\n      "clip_start": 72.417,\n      "clip_end": 81.778,\n      "video": "media/01-0033-source.mp4",\n      "current_extraction": "media/01-0033-current.wav",\n      "baseline_text": "Hey, so right now- I\'m hard of hearing.",\n      "prior_notes": "other person says \\"OK so right now\\" and the target says \\"I\'m hard of hearing..\\" at the same time while someone is talking in the distance.  The target continues \\"can you come and stand here because I\'m hard of hearing?\\" and the other person says \\"OK\\"",\n      "extraction_identity": "mostly_target",\n      "target_words": "im hard of hearing now come",\n      "other_words": "ok sir",\n      "speaker_confidence": 100,\n      "notes": "",\n      "basis": [\n        "voice"\n      ]\n    },\n    {\n      "exchange_id": "02-0064",\n      "baseline_index": 64,\n      "start": 130.548,\n      "end": 131.208,\n      "clip_start": 126.548,\n      "clip_end": 135.208,\n      "video": "media/02-0064-source.mp4",\n      "current_extraction": "media/02-0064-current.wav",\n      "baseline_text": "What protects you?",\n      "prior_notes": "the target says \\"I\'m not a vendor\\" twice and then \\"The first amendment\\". The other voice says \\"What protects you?\\" twice. background music",\n      "extraction_identity": "other_only",\n      "target_words": "",\n      "other_words": "what protects you",\n      "speaker_confidence": 100,\n      "notes": "",\n      "basis": [\n        "voice"\n      ]\n    },\n    {\n      "exchange_id": "03-0118",\n      "baseline_index": 118,\n      "start": 206.596,\n      "end": 207.777,\n      "clip_start": 202.596,\n      "clip_end": 211.777,\n      "video": "media/03-0118-source.mp4",\n      "current_extraction": "media/03-0118-current.wav",\n      "baseline_text": "Because I had three people come tell me.",\n      "prior_notes": "Target says \\"You dont even know what I\'m doing\\".  The other says something at the same time.  Other says \\"because I\'ve had three different people tell me\\". Target answers \\"That\'s hearsay\\".  background music.",\n      "extraction_identity": "mixed_speakers",\n      "target_words": "listen",\n      "other_words": "nothing you could tell me",\n      "speaker_confidence": 100,\n      "notes": "",\n      "basis": [\n        "voice"\n      ]\n    },\n    {\n      "exchange_id": "04-0157",\n      "baseline_index": 157,\n      "start": 285.463,\n      "end": 286.264,\n      "clip_start": 281.463,\n      "clip_end": 290.264,\n      "video": "media/04-0157-source.mp4",\n      "current_extraction": "media/04-0157-current.wav",\n      "baseline_text": "I already know.",\n      "prior_notes": "other says \\"that\'s soliciting\\" target says \\"Now Im asking you I already know... like I said\\" other person is talking at the same time.  I hear \\"that\'s not my job\\".  background music.",\n      "extraction_identity": "mostly_target",\n      "target_words": "i already know",\n      "other_words": "",\n      "speaker_confidence": 100,\n      "notes": "",\n      "basis": [\n        "voice"\n      ]\n    },\n    {\n      "exchange_id": "05-0215",\n      "baseline_index": 215,\n      "start": 402.094,\n      "end": 403.314,\n      "clip_start": 398.094,\n      "clip_end": 407.314,\n      "video": "media/05-0215-source.mp4",\n      "current_extraction": "media/05-0215-current.wav",\n      "baseline_text": "For a formal trespass, you are.",\n      "prior_notes": "other says \\"For a formal tresspass you are...\\"  auditor says over \\"you are\\" \\"there\'s no ther\'s no crime taking...\\" radio in background",\n      "extraction_identity": "mostly_other",\n      "target_words": "there\'s no",\n      "other_words": "pardon me",\n      "speaker_confidence": 97,\n      "notes": "",\n      "basis": [\n        "voice"\n      ]\n    }\n  ]\n}')
(WORK/"voice_embeddings.npy").write_bytes(base64.b64decode('k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE3LCAxOTIpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAryYYA9E3/UPKH0CD7MhyU9xdnRPW10mT0daXQ5ahlPPYpuob2c04O9luHlPP9rbz0a46y9mgDPPZknEz4Beni9hyhlPfRCj7vrA4y9itULvslpAT3tH489TvS/u+Hd5jzRNBO82H4ivTIZjT2+seC8MHu7PGww4L0p1yK9fpBDPaf23byFqDA8YlwqPkJlo709Ssq9GN2uu4+ynb0Y7jE9+04PvuVCxD1XbCI9+0VxPbu5CryWeby9nGhEveePpL2thSK892Squz16Zb342KC9a41Vupy3e7ysHRg+7RnDvC90or0/kBa9AkjIPZ70Br6Tx2+94YKzvcES3L3IAsY8gNXIPT0GJL3rrMc4YlCWvT7HKz2LBZ48uD6+vCi50z0UKky8ZnkpvQQj/T3sOo29Mc1lvXhh+r1uyXM8Q/NzvcWvPj0m+cO8TrLZPdUlwT0dK8U9OVK1vYpkRr4/pVQ9icuiPWuRXj3Eq0O8Q35CO2dyKj3SDng9z419vZC4JD2dfJy9bMx2vbXUJj5VW/k9BMs3vRv7IL1nJw497TaXvNuos7zBfGC+nI/HvfX0cb0F+RU9IDYrPk3foD3wpra8Rr3fvcpDgT0Sd548KO1TPexPTT3EpT887TLZPD2PQLyW8MW8JR6LvdEztj02oqY8LSyCvUG8Vb7mere8HoOxvAsriTxRA1a8M+2cvJHnnj3JRL+93+1BPYRVmjwAmQK8OZCHvZu2pT3b9La8MUacOo6hkz3e8wY9jPmsvZuOyL1phi49VDESPUjaFj02F0A9uIjevHLjA70tptE8pgjQvOI8lT2PQW49FMBYPQgBWj0gOee91YDXPQOUgL0zNZs9A3G5PeWCaDxe/uk8nnhLvfYIZr0IGD293YSyPYX5Aj5DguI8tuQZPBEfhD28CnU9NtdlvLncEr6aXsu8a5miPe+xrT0p+LQ9AjT/vCKTKz3ZUFs9U4cXPMgDuz0aF+g8WmQMvZJFgb1xtAW9Vu2+PNG21j3czgG93t8WPQmI+TzOs8c8mPqpuy9L0T3Yrz09Hs4EvZbblT3UEmS9OfLoPWwsLr2Wcti91u5OPRT0Nz1f9De9AMwjPTLBHD6k9RG944kIPkGjhz0iYpa82gPaveyVgDvmCgM93AaAvFgeTb3J4sK8jDYovTDMVj27gvQ8YJ66PfuypL2KBFi89PDHvLUfzTyD8qw8daZFPoO8IL7HlzO9DMeXvFnEz72p7Qm9kbbyvY7jyD3hQg09gglqPSGZqTxYoGy9RJc5PBinkr0TjLM8KD4rvZ1LBb5RW6i9uLayPGWTDr2mqvA8nJ+ivYYZKL3ZDUu95KauPaH4p72mSkg98LahvaL4x71mPpA9euNFPaAgnDyHiza9S4/OvNa2O72IT4s8K7H8vE4miz2HMQM95iXsvd0hoD1pzqS9OLl3vbEc3L1dsoc9pFvMvD950T3m1HG9V8P8PTSYlj2+cMU9qbWovUqfCb5iEqo9m3mRPULHsb1y8P+8g7IlPSlOxTxuIrw8sIeZvbf1KL2r4vS8ziSUvTlsdT4JuxQ+PBIRPQ0xs72nG/M9nMwDvGoUE71utwm+XtASvi+rtb1mzYs9QhEDPh+qCT7kpSW9QPE8vbWewjzudd098keHu/swBL2XTpo9uNqlvHorZL2WgQW9YnkNvvAHiD3VYpY8Nh+lvYbdAr5Tc708ibMFvVfi5T04zhI9czzEvTl+DT009Nu9PBSTPe12YL2mKH69b2oAvbBVcT09ylO9A9AwvMhHSb1/RI89HROvvVdiVb3rd5M8V5d0PeauWL2nge27d6+MOzg2OT0lNDU93+Q0PBRL9z0WCtO85gEoPUw5SbwrNK+9GcOmPbMmw711G589NRtlPWSrfrybA7k8SKXJvGWUBr0YJIG9nG+ZPXzDtD36fpk9xrZjPD0Bwz1U+YS8MvUEvkSmgr1WArS9dBiOPel8CT0nMC08zymovPzwsz3DTo89VCi7uiVzJDx04k09vG8qPLkBe70HpUe9Fq06PRZvoD1djpi9EvBaPLLuJD1TJnQ9dfImvDx2Rj0bGZc8XDsmvIXusz186729+MUVPiGDAr202bC9TeQCPt5sUDyhhdW9HqNjPSZXmD3H9Hq9c/DIPZBzFL3/fH68L522vRwRpzxIPUE9RXvTPPQ9Bzyx7+Q8UC5EvbfWGz6qWT894laiPRJOHr5vWgu7kdp/vPZArbzaq583csKCPgVz1b2SSoi9OUGBvQyZpL2Lo9K92xoovlVyWzz5B6e8ffvyuVszID15xju+Sh51vfuuuL1I5Vs9fL95vf+6ir3EyEW9ZhQdPKTktrwi2xQ+vdG0vcjGM71BbgS9w+rlOvsyXb2O8UQ8QxWyvAcltjy6gOg8m2lWPbr8e72A3DY9w7cMvQ8Ibr0XipE85L4avHBQTz2LeYs94LWrvbQbrT01f+W9PTkwPSBjmr1Ljym9fdrovDjsXT3agNe83W9WPc28sz0U4gY+QQ9BvWuNOr2N2AM9UaZUPQb7mr02SC09SCQYPahTaj3by/Y9g2kPvnPl/7wad5M9xzQ3vdd+Bj4f6XQ9rs6NvbePPzyKrOs8cU32vW1/grw1xSe+HScGviBeML3Fbtu8YUupPUMrAT7Q84496mYTvhs2/zyKjfc8DxH8vMxsez26PBc8Jb8ePd8zv7wk86U7ArRPvNiNj7z9xxO9QJTYvZ0D8r1XS1e9Nr9EPKPKAb1iTYg8c005vYL4TD3AMcO8Ogi5PK1XijyDrHy90J+EvQIxPj03Xe69jHbFu9UfCz0+LnI91eabvbh+ub0gG4A9CqCOPbU0mz1FNSm902ndPIjypr1ymwA928lXPa/Q7j2BB+o83EibPZ+eBj7auiO+Ud2CPfnDjL0pzrg9pZvKujGVwj3sPH26BqP3u4pyFr2eKqK9elJROx1YkT0j5z49EklLvbvVGT6haU49Wn2wvfDEfb0BPGU7gsJuPH12Oj3k+Gc9feLYPP6dHT5KZz89+iK1PDLwkD0W0iq9oaD8PDHVuroO8ae9nWfGPJJ9cz23edK8pyLDvHUv7jxIQA49BwEmOlXBkj2gVo09chmmPBhf9z3UVZO968VTPYBJGr6fE9y9CfLzPX4eDj4vkpy9DF3xPVZvgj09aLa9OmnaPfb4L70MggY7Ge4uvvVlET3CM4c9C+A1vL5RVL1zXcM8cy3oPEz3iz06VIk80w41PfyZJr3xgJI9xE7DvJ2jCz347Lk8RPodPt0sjb2m/pS9Q+govH4tub0F9IG9OfEOvixRMTwjBRy97k49PZB00DwFWuG9L9bcvCDrh73ii+E9TjxevdsvS7ySfzG9wv5vO9rGpL2PRrQ9ChKvvQJQprxz1qS8WS3BPUYAsL09Uma7E03Ovd1u3713Ux89vk6NPNKWmbzbC2k8Ly4yvcgGpD1uIje9XgWJu84noTt67H48J7AAvrMRnz3U9eO8jLCjvEvcQzuvckA9ci4lPJXUeD3/jI67P6LjPZTuTj3UYTA+035pve4TIL6Zo/c9DbSbPactl70/EKw7yFm8PQEAhr0gFn08Ts0vPPjrPr1sLJU84BNXvbQwTD5GoEQ+HhcOvAxsbbyNzxY8ykiNvbjXi7xWU6K9YphnvH8AXL3QGTq6JxktPpzkhT0ve2o878R9vbqTTD0Nq948drCrvWPaWTy+iaC8MhWRPelfvzvoPx29yqmCPO8WULzpmnw8+eswvYVORr6nBac9Si98vA38Oj0VqBU8xYQSvisddj04r/29XuCGPZrm7Tyzemm96BQtvdA8oT0IlMc8kfpAvG4Hqj3xSNU9tq6lvRk/a71RG7c9Ue4GuzJZZjyliSO9OO3yvIFFIj1AQ3494/gAPVww7zzxX4s6vBBzPdvBxD0/kzC+v+ffPazcCL6Po1A9u4aHPS/9JDxLw1W9s7wIvC0Pl7yc07G9NYmkPSNcxj200Xs9DbIqPBV9gz2x5K095oDQvepU+73SI1q9yQibPEf/wjwFXQk9mhpLO8l68j3gasA8+PG7PIhQWLxJb9k8wAFKvTGEjL18Ghq96lYYPQXl2Dw+I1E7qeaYOgKMH7xnscm80llsPWWrhz2Doj4962znPI2A2jzxjyY8azQwPW4yQjzhs1q89OnCPDIsID6twyS8m7AHPlhibT0B/4+9COWbPYhlBrojv1S9WTrVvdFmaT1H6wG8DIWOvVLu4TylOyO9n2HFPb2jQTwAzRy9zfDXu0Em4L0qAqc8c+pVvStNBTtFoJE7+S8wPnpfBL5ZCjI8s9R6PcGySL2fCTe94j1DvsnRxj2opoq9FgKSvNs1vD1iHhm+jedcPE+IPL0uiq+9Mif5vUJjrr133UK9raotO+Cupzy21v085hwevrrAjr3k4y+8o/KZPcXa8b1NDiY9FDIJPbKdY71yc8c9S7m/PdKjRb2hijI9fIqMvQrkBL0eAVy9fg6iPZjfwT0BxAI7GaUwvk5XFz7cGDu+OKnAvTKi3bxiEK68gQXJu/8ZjDy7zO+80f7EPcGMtryERJ89Ufe5vdEk073keHk9v4ufPXCMz71j5yC5PvKyvE7LTD2Wqry61bJZve3tdb3Brki9vVfEvQO7nz1BlbE97kGLvHbCr7zkYFO8uSj8vOGLpjxZpBu+Qu88vZCynL2FMgC9hOjoPaoH9T1tO4e64X4Hvq7WFz1Wc288OQqQvGyjyjraIzW6WUeQPRgHiTwFfT29xxOevWcIfz1zJ4M9NlDYvVY0fb67Dg69R0FhPeEzQzz11+U8UqG6vbto6j0ljLa97COavDuuKj0r2Uq9Opx0vUBYJr0GoTC9IIuDO9PXGTq14bY8jiPjvdePrb1lc2q6A/v/PZI9bL2b9XU8gu6gvWs0hLxebmE86YQzPfsCqDzBBJS9mVF/PP8FaT0XUpu94cpvu26z6b1umjg6fClYvR+W/bw+gCK9nHtAvRZ11rzkEwa+XHnaPemZvD36UNQ8GODDOx8rBj1yYlM9PjfCvYaWLr10Qlq9IldpPFycGT0adNQ8eST0vM6IrT1/Vk899hHvvN3kB72oe6q95mdUPUhGZL0FaDm+FUaMPBMIj7sZarK8h9yUPE92IDzyGN89xUOJPVExeTxeiBQ9n4lTPVqd/j3gA5y9uc+QPf1v2Lzsb7a8LkEcPsHoKz75L1O9MlLMPXlxuT0tDiK+ELcPPmGZSzuCfui99I65vaIQmj1ejKY918KIu3z31DyaOei8Lwztu5o+Gj5H9JQ9fYttPU7zGr6dvRa851XTOmK6Bj3qaAq8Aqo8PndYyb2/lIO9OOJmvBCwhb0cDyG9NgkCvmhN4D3pGWG9OuIiPXCoKT20U6G9x9EpvRxQNL2RuaE8SwqAvaqIAT1XjS69PHIaPJm5or2nRb89udurvcCBqbw63p882W3CO1ukqb1cyie9RaYNvDtRsbybHgQ8bAukPLUwWrwHrHM9C/pSO5sPyjufq4Y9pGVKvJVCkz3VTz89oQHWveEF2T1tMci9f91vveR0E75zjMI8/2eUPA7SubukPR68J4yfPZx0E7sS2Bc+tuc/vUd18L2cNRo+PEPgPOmkOjxBsDw9g97ovFEnNj1JcCw9WrG9vaOqATuKR349dhGvvb39Dj4uwCY9y/FWvanQaDzhz8Y9KHbWvYhX3TwZjx++a3ELvkkWAL1U+5C806ESPoBCCz66lAM9vvbLvYN/JL3RCBA9pBDBvRvfUjyr67o8pVOePFkLHr0V2z+9Wj7cujx7xT1yTQI9B6M0vWqTI77l0pu8vuc7Pf+Ukjtz/fI8k7VMvebjzz1UMBe+lRSyPCO5ej1NU7e9XZYuvUbDbT2541y9Q4/FPDMM0rk0o/26V9KovUqI1ryoBA49Af/kPXOeOj2DqMI82ZJgPDTrfb0ty7M9AcrjPF8h2D3CYsc8ekEPPDHYwz0Kf7+9E8SROgWfWr14K6E9IPjQu73bAr1G2Je9GeiZu+cHL73NkAu+dCNVPS8Mzz2CBGk9OVMEvC7obT1iwZW8BSmnvOnljb0Yr4W92etYPN2Fdz0g+QU+fjdBvQRejz35m589XxmkPObrBr3/WLG882nmPePJ8bt74628gZ0RPZOrKD2dZ2O9J74GPRCfubvJsGa9gw0XPKbuaj2PbGK9PrJ0PYXrAT43mxi93oUePflmEL1rara9DhjaPYKf7T0L0iu9OeXsPXVn6T2xAIm9KcyAvJLZFr3Avi68dQILvoLlgr3ZbOQ6yeNZvfaHlj3BBYi9oQRxvZjH2T0Uv809aMpIPel69r2V9Sw7NgO7vCRmrjygY3i87PgUPml9h72O74u9cleBvBnmoLxjEIq8/eQWvoK1g7v1U5S8u4CfvGjxwz0yy9C9oh9JO8gv5L0cZ0g9w3+nveAGjr0CJ8S9LMG5vCgShb0v6LU9ZUPPvTfVNr1vYuI8Td/ivCfSpb3lkHc91f6YvS8mtTwXSAU9QHRUPagKGrw+Jzk9vagDPRa/krwUDrw856psvJmGBj6PXLA67qASvkgayz09I+K8ERstvA7hPb49pyq78rVOvQzKqLuHZZO9Et1gPb/oOz3+89o95uTWvTK8YL44shs+bVnVO6aQnr2tVZO8oXbAvCM81LwtDD099TQOvX59bzkhSeC8ESOkvUZ2Lj4DX849TAQBvYWkn73ZmJg8JtfmPBqEtjvm4ku+e1hIvUO93L1mAy47mOYdPhsJDz6hsnO8uzgGvrcGGL1OejE8X7jsvFTrRj1G90E7ap+WPb3DJD3mcdG9wLBxvHrf4zrJ0oS91d2hvYRw1b0dnzS6BjFfPIuD3rohLBQ9N9cFvRBgDT0TR+i9BXDPOxfbhb1LBxC9lCalvfDQ1rulFXu9fP8ePYBaAL0VveW7Kh+wPFEfq71aqEM9BKnFPcQFmL0BnbO8sxaNPMJNw7uNcaQ8QEHgOzXHrT0v9jk9yd+RPADXtT0GCti9jm33PBs2AL4xh/Q9A1cwPTlSN7y7dOK9G3C5vGMcLzyt7du9r2LLPfppP7td7iW8GgIkvLo6sjsY/7s8J0ADvnwgm70nhvu9KckYPWt0JT3vo0E9UsFavSlx6j1rTPY9eo0dvb85JzoGBYE9tjQ0Pdw8CbwRfEK9V19zPOHarz3zyMs8CQLmPGWHgbwnAn0928YpPWPb0D3mqrS8PY31PLiIjT0a4vq9ka6sPae9Fz2p/Ni9mp27PS2g0TyWpnO96YEZPjhcDz6wqmC9znIHPs52STzZkei99gqdvSoJ/rwFR6s9Q1K1vOs4gTyQmU28H4byvN+hYT2AD9I9trTqPc8SFL5Gtl48YVTrPCK34Dwtxx29RYwZPoVO7739jwm9m9OSPOe0570YlsO8DMM8vvAYHD2hdbM6YfeYPG7L2j35Nsu9zOoEvucFvr0j3bm9uXaQvKoeYb27qqC9lP0ovM27E73Ryo097RSmvSZ9h71JdAq8KUetPBqkgL1o8pI8+SeMvNlgeLyVyHE9qlgXvL8ldr1Twls8nh3uvXdcq70WIPu8BiKBO9aSnj2xy7w9n+sFvg7/IT7Qjpy8JNZ1vTnKq71LMbC8LM72vI3xez17G5y8HqbkPXMDkT0cQ4I9G6ynvafgRr05u4c9IJjhPRhKSb2P4Rc9ugY3PWbETT2To4g9iaStvZMnI734GzO9MVEWvdtfQz4j/hE+mwF+vTvueDySD6o9yzk6vTp0jTz78Ai+UeF7vScMBb4v2b88EsQcPkq37j1ok9u7c7ePvYmqYLupuhM9cVonvQVtozu+y208rOCFOyHjGb14iGC9qBCPvNkuUT2Mev+9xH+UvS5RBr5XrXu9vaPzO/ayozyDe/U8KYGMvSq8CD0OOzG+jpx0vXNGLbxiHZO9PPhfvTPQbj19z369Ik9bvDIQqjxA75w8ReYgvBvOcL2t75Y9iJwBPkgg7DoTPH291BqvvDPzsrtG3Hc9xW5qPf8toztwpP87kuTbPVy00zzMum295LyDPSyK4L21HQg8w1m3uyBFIj316zy9KN1+vQlNiL0sC9u9EYmEPUV5BT5os/08Y5aJvf6X5z220A89nzcCvv6hFrxcuZ29m/S4O2eKOTwg5xC8mJeCu2Xx5j2dsxs9oRl7vapRBD1kI7u8bGakPL5GvLzj5Nm6EPV6vIpD6ztNzXG9zFH5PO5Srbx8bQ4+0QApPaIbwDxgWko95tlcPXMSDj1kr5W8+pAJPiwZHD0qwsy9KxVBPZuEpz1EDAK8POzRvHEmUD2vg769R8ObPKYzLTyotow8FNDIvRFxij2agD299uREPf9/u7wjZ0W9CBBYPRAf7j0EP9c8vmtXPTICLr1slXW8iA27PIB2Az7ryiq8C54GPrpRG74lBE49iZ6fvOhf+zwxWRm9COkUvnfy/D1DQUI9gKuiPBEFTz070W28iNwMvlBX6j3+xYa9kz3QvaS89zyIdpS8wkDuPNmrijvdPsm8kQgIvsJevbs2tya9QAjPPc1Yb7s7wAY8pyyaPeTwwD0AbBM9x62dO4PFeL19kW89v7SYvDfZqbyS6sq9OC/dvLoE5DzrELK7TeCsvHoilz2bn968nqFAvFgUR70ALxA+iPVxPGPjKr3liGi9Te5MPPabrrqNBng9SZhmvV79xbxI4yk+fD7GPT69gr3mDVq8enMGPammHDs2tPY8oxlbvhHDvr2FTsw7xUuyvTuJ2z1kC6I8jQTJvXnKoDxST8U9IRluPdObtD0Imrg8ACwbvsCmR70nuey808nuPb3ICj6aoJu93IIbvRXh3b3eT4k9XMOLPeoZojxvdwa9rcXAvF34srxzWSc9XnWjvdxOAz5tWog6ec0LvmLg1723Fr28NM/fO7JqgT2mOY89K1ywvKkVOz4RoVI8Bq/CPUEyIj3DP8a8pSaBvBxpdDwxSQy9JazIPebWhr2emZo8BH8UvrQEcr3E8xk9PyS2PX0Wlb0rXl6995IUPW7tGz33RjI8+bFqPfgQ0z1ycsC9O6GVPRthfz0Jxao8WgYZPC9fLr7Q4Yk85trivTYv2zspG7M95q2hPZ/uAjzRSKU7biTuPfddhzwUWtE8lnJXPFmBaTzplgE+tkAJvhJ/ZL1OzMW80m1GPLSMpDzOpw8+qkLcvUeNqj3OGpw8HTYbPWUknTzB8iS9fnb2PRMhxb0ZHMy8whTbvK4nRzwpWQk6NYYYuzUnMr0h5SE+Dd8+PUyJWD1KpSc8/7s3vC8fCT6JA7q9SWlRPDCmHj3TBpm9JKtVPfqjwjx3LGk8DqloPQbx4Dx0N0W9JaH/O22yxryLBLS9rpWDvb94AD1trhQ9XSgUvcKb87wKqPU6JRTjPEJSDT74FCk+UWngPKPBoL1PFBw9j0AfvETGkT3bDOc9iVcOPuZwNL4cqt08kqFou6iQRb3xZK+9boPmvb0llT0XWuu82xmJvRvJ8TyCYuK9TRbKvfHhAT11fkC9Xo22vYfRgbymypO9oXxvPf1e0L1zBNC9Y99mve2zCz1d87q9bpuyPXdUED5ZOf07qMOsu8T9az1/fzk8GhJ1PHLWmbv6w1U8a7jHvY0ugL3+m/C9hP5/PA84Vj3LfNE9yfa2vQZDkzy8Z6e9BPYTPXueUb2k6+g96GRhPRH6tr2v5uK8T7WAPcHWIr31nEA8v757vWy6e71gJrE9Z7XsPexP7L0glDU9PxiavLbXyLz1fV689A35vZPh8713ASS9rsUSvnyhvj1Fbxs9CULoukrJzbxJ5j89iqMHPZzatD1rTJG9d08LvkTVkL17ixu9NaClPdWWAj7h9/Y6OS67vcwx4LsnrrO84fUPvdJDVz1xo6I74gNmvWiPtrygkTE9ZP0CvkN71z3mZPM7J3fdvbVCGb5P8mu9m40CvQiADz0PjrK8TcnZuouUMT5aIZO9yiNePMYxir0BuRW6mrDCPK8EozzW8vS9fMECPUWqHr0wvZO9Ovfjvah7Prxf9ZA9JNf8PQeZTL1G/Om8TNduva+tcz1hFw69VMWkPaVdJj4b2x88Mht7PdJ2ID0rlp87e+fNvHiYqr0DzFk9r8JOvbK9tj1NIdo9bRn3PH8j97xf2aK9hxnjPYMH6bqKIQk9N8MhPWZSCT34nYo9myscvubBsb3Zsgq+1NDjuk4+VD3LVaw9bl0pvdwvGT0RVLG9bCPbPF0oeD1Bchq8VHBmPcl7AzzM8I69mGqJvXb0Izx//L28FqNcvf3UFD04RAk+mMECvAaLED3PrJe6ERMsPLGaJz0agQO9UBhtPRepKj1NctO9e/tvPIR2CD2w7Pg7lw0ivbrc0jyp72a9hxGUOwSLvDu/Epa97dfFvaT1qD337xi9YCrhPMtEDz2o+JI8GeeVPVuk1j17jh891F2OvN0Kcb09Vbk9vTajvVkVCT5trT+8Jf4/PtbOG76+cs08WmhYPXksTr0Xg6u9czMUvid1hj0+ZIi9SI+VvP1LpD1uWGq93kbDvewMtj3x9PG9x80Dvs6HJrw9fCG96lvdPf45w7xPlx486M75vTQL3jx8QTK9gWkFPj0WLL1Ul8K8Ii4RuvyX4T2v8G08XjiTPat4+bxiGQE9XyfNvXh/Jb3hy4e9DsCVvOSJvLw5b6u9G2VUvZA7AT44Ht+9MZ+2vHzNML3mye09zMlfvftOAL3cMLS8b53APF/kiTzOzlQ9kZyZvBDa2jwVoug9FieKPSyoLr7xpMk8FqNpvfSsCDyg5v09VHcnvkWB3719xwi9rhD7vD4dLT6gQU68xQzCvW/N0LwOWJM9Hfr3POXJPT3oqMy92pXZvZaNrb0ujKy8BbjZPWenID6W+/U89AzcvJIEfb2xpvg8XQ9BPUlMqzwnJPO8PGHevLBhi7375I49ONgAvkSymT120E09uyuGvQ9iwr1DYUe8ofuRPCvluTuLcV89/G7HPMUNFT6R2TS9cAtYPR+9qzvTHuC88kf/PBRGhDy8HY29udwAPo5qPjuTsvS8qlBkO8sjDb1NA0g9QMMAPti9Db132xS9ExYQPSrnYzzbAF+9udPUOsKx0j1RTSU9uqmPPZGrBzzg4Nm8aODQvLaXRb716kg900aNvei1pLwRGnM9c9ySPWIZWD0uOxI9WnnyPIogoj1v+uU8itIiPT24Zj0H9Dg+eMCjvYbefjwR+gi+IHb0u6dQNjxEyvE94RSwvSBquTyinum8qVOgPPpHND1d7Im9cwSgPRBgN71FK4S9r0kVvlzEFTyDvGc95bQivLDVHjxE1BM+aAgIui1WRD3AOnO979CPPUHjtD0j+gm98B9HvQ10IrxTL228d6vTPQ0Y/jzkU5e9cI6KvTGTMb12sYy9DV4LvE4WZD15QCe+CYjDvUPbuj37Ssw7Irk5vD/YAz1WSNY7OhHZPW0pmj3mzFk9FvFuPetF/rzdgWU9fQ5Vve36oz1ui007WBQaPlun972LQIE9Sz6CPPU7mbwfydS9QYMDvmHEgj28Mta9tzZjO1ZSoDuaF2e6jWr4vavo/D2FUsi9W+SCvfBcOL15s868N2wcvNkYtLzZbEk8dnjYvWi0ortlF286luXSPeoC0TtXvzS8M14FvcBlBT5WGM865IxvPQlhRDzkUW49xSaFvaC2i7yYoYy9qBDXPNdokjwRvfO8uICru1LkpD0Zw0m+RUTEPOTN/LuFkRE+L3MUvX03Er3i5YW9JhiPPdiEyD3KpBo9V/jrOn9QTL2L7DU+uEtlPYN7G74TXQA9bxGrPNHtiDw2rQU+WWAhvvd0wL0Q9l88NLmJvRYZKj3d/nE9qFAGvsoAgDxE48U8kCsHPUMOVz1x/IC9gdgZvnkYgb00wJo80lDUPZIe9jwv+OC7wMycvU3pBr2Bjoc8GdtSPcGOKj1tCMi8Q2divVNbAr0LhW89ogQ5vtEWLD1AxUw8GXPDvSkFW77rUJI8Y19jPWzlDr2ZjBY60HESPRyLkz07w/G9ybUEPh8tYj3Iyp26hGAIvYrTmj2A2ya9MaMBPnItkr0gN6A7oy1WvUKNpbttAPA8O13XPe5kSzzhpTi9pWjbPAFRjT2gZMM8Jjc8PQMi/z23CN08+EeQPVii6rw77TQ994jbvAeYBb5WjMA9E4OXvfCvITz1OgM9UPupPYr3pbwnfHe9Cwn5vFZEPD3yGGk9EWaGPNgs2jwS33o9JE4ZvvRJwbyXrg+9AOiEPdzNhDvbv+o9oRbAvbTyRLzmOJK9UvwtPdETBz1O7xu8gPstPXV11Dz9HgG+pJ3gvTFsHD2/1TW88anyu6yjtbuyfv49bnxBPXiyNTwk3qU8W23CPNKYNz1vyzq91WsxPeNl9ry/WX28H7SHukBgMz00FWS97DjdPaYE9TzFYvO8iH3wuuCB2rsSV6u9NPBjvSkOoT2IGts9i8ikPIxMoDzs9Sk5WbrjuR+ETz3Qgjk9OIRBPTslf73NMny8YhWPvT68hj1T6Qs9vbXyPWDiWr7sdtU6nHRIPO+7mr1qPlq9NprCvUo5kz3PZ5W9NW3COz0KuDw4MKK8w9oFvmN7RT1HJaq9+MNsvQCJmDvH/R+94lutPcFPijzqYkS9V4LKvYo8x7xvjk+9WxW/PSG5Jj0IuKk8qCa+PeK04j0U1BM9CLWbPez+Cb1gfK08MfbbvDW3TL0qF6S9kQzxPBRnkz0kqhO94MKnvdV/oz3h2hC+5DURvI9bZ725NAg+Ql2KvSv0n73PmqW9OhQVPUPfIz3FYJA9wSVQvC43KL1/ZP097FzIPWTA3b0Syss8QHVUugb3yzywf7s9wXcjvv0SDr676RG8uwnzu9t8Az4/oJo92aewvVhsvjxLU6C8a+RbO28DHz0w6Vy9IKUWvs+Ds72KQPG8UgDbPeBeCD4nbAG9PIzbvZHH0LzzDZ28w2C1PFQy1LtXhVm9meHjvKtzpb0GGac9MXH0vVS+WD0QH149v6CNvc9tMr5+DL+8rjN8O7G2GT2OGiE8T4lKPSOUHT5Cd1W9S5nFPQNq5jy9SqS9VlgEvSXUAz5mq9u97/gZPTLhnjrTB8e7ZGTfvdL23rzWoIo9ceImPgaXDLwSo2e9qCbDPOYMrT31sQe9spJ9Ou1NoD3bOUE8m9rcPUl4uTz1aX86G2vJvSUZU74V5yE9F+K9vVGwZbw/NDQ9wKsmPVaKeDxwkD88H8uoPGtspz1qBhM9uD8lPbUmoT2KcuQ9zX0Zvj1nib3AeRG9+mdIPSOsY73jPPw9qZucvTz0jD1A9g08GAUZPKOFiT1IZkC8II+FPYv7fLxR9pm9DSksvv3/UD07/gI7aCqzvNhVEbxczms+qdEDvFUetj1lYDs944LAuzlL0D3uTOm9SyMVPSG7lD1o/ms9w3PBvBtNKbzGV7M80aNAvFEoKT2x4gK9ieQtvagsZT0txgK+whE5vfv4hT0EQ/a8f4GAvHvsCz2S/+u8EyqUOy46Az5xxKw9DjYCvR3b6LyPT0I81l56vVcXLT5HZNU99smyPetKHb5NMdc8dNp5PGGFa73UIa48S9qFvRxNoD2Z6Q68t8mNvSdqK7wDNu+8tzzHvV9Xk7zDp7C9kxaIvOtp6rsP5rq9aTtxPO7kar1Y5Cu97PbovSOPDT2uqQC9qWDuPcutvDzJ4YA8GRzLPKi/FD6kIO08fXyNPfOcyrpaHd07n0UkvrGdcLypY7m8nGFSPcy8YzyeEHK9uX7YvQceEbx/6AS+Y+NGPdXyQ73nPvs9/OhjPDGz/bwuD6m96yLKuhypBz2+dp098ni6vL+Onb3MQbc9FJwIPlv68r3Kg1Y9S7qmvP0QmD1G5vW8P6qYvUjtZ739nz+6AbY/vQD9FD7gIYI9Z9YmvXyaarysC3Q9/MO4PXE6wzwFlwC9TxkRvjXlpb0+Gyc9bUcRPb0MNj4KdD68jKmnvWBXIL2Z4Pu8qlWiPfdfPD14pFc9oQlzvVg6ob2LQWU9uNKpvREHmD07rYA9bJCcOQ6VHL5roJ+9TaQQPQT53DyldFi8LLZEO5FiQT60+Lu9Q2fQPA66srzMnqe9hXcEPUfXMrxWK9S9PNG+PL3OezwhrQo9/BQMveGPzrwXoOE88QtfPY/ljL3QYWe99kyvPIEAyj382oS8yBqRvZW33j3JiW686NW0PZXRSz1mDUa9n6cFvuWjDb7yqKg9M2e2vJqG9zvpfW87RUrvO8hi8DxuY1W9KTScvMeIybrmxrE9kjXlPDePBz7wYao9WzcevoJtZr3Sita9nUxjvEQxwjx1vvo9bKxqvXx+Kr3cPJi9hewFPO/LBLwtvYq9HVf4PPa2oD2Djam99LUvvq18pLx6DQe9UKMIvQUgoLxu9b49AkcrPT/ICL2l3Q08ICRXPSqK6T3yahu+2gCuPR2CojyKV7e855qIPe64Yj2fo4O9BLBUPDooXD2jfVq9OALGPPl7jjym4Iu9lMuMvQEvSj31Y309kDLjvIX0Dr0x/z66cwqhOzNCgD0Jiv88c58fPdhom712Gpi9muY+veIejT3c97A8FP0PPnJT7r2r0gI9nFxKvHeO2L1QuLa9BYSIvUMWuz3mXIm95EM+PLbRIjzcRcm8hQEKvnHEOzwnYmi9phZwvYFAhDzBcmQ9G5ybPfff/bsowNY8b6nhvZL6KryRkZu8csDQPFbTgj29N449lfyGPUwqzz26R8s9ELyLPTzNrL35I4Q9l2jxvA/a/r1DuBq9K2Yjvcb32DxIkiK6H0TZvdytpD1VGR++EMsIvLrDA72ug/g9KHgqvTCqXrwhSxS9prWBPf98iz2ODWA9OstQvb07Cj0jbA0+6IOCPTf/Hb5JppQ7s2LYPKvm5jxwX8Y9z9hBvrDy0jsg26q6KbfpvUhTiD3FvBc+wv+BPfzUhT02QLI8v/YxPTZRkjwbw9e9uNYjvevIrr3KbTm9uj7gPAjGaT1L1448gIaDvQ1GPj2bil49KJegvPaf5DyCqji89estvNh8Uz3/OCS9VLAIvgWStjxSUCe9R3JnvddvWr6WYf697WCaPaxibL1gKKo984c9PcBtoz25VCi9ZMq4Pb8XkzxVZqE8+3usvWkmJjzIPtW9R4j1PdqOyrzeiPw8pe9uO6fxnbw8/vo8S+3aPfCAer3mH8q9+qCuPYslAT0tIs47QunevDBFKj4AYM89KFdXPWgauj3alDa9pJeGvAWgAL7pprs9NvNGvTICjrxK9cA82K+EvaaWory5zqi9feqcvMoBHT1e9yY9bNhgvM6uJT19iiU+6RMkvnI1g7wbJMu9KKcsvdNuW7yeJbQ9avEnvbx51jwdjbI8g+1wPYMADj2mLGK6y1kbPuwH8Tw8tx2+CFndvfATGr2XhZu9zeeBvP40PrvsfIE9g43TO+zC2bzV7QQ7YjoRvdIkqz1JsY28hhgTvG9P9Tz+nz69A5KKPQ9ubj1r5ow8s/c0PRUPq71tD6G8akXOPFnOBr4q9sC9iagjvlDLjzyaDMQ9xqoTvFp07z004iA9UN3wPcBX2j2iHXk8mb//O7T1pL123je8TnvxOpZZOj0aZAQ9cQIjPqleQ72GdMI8jsNLPTuT+LyKOja8mLjKvVDGmD1T2Um9aTOhPcGvJT2QtHq92KWpvYxmDbsaIMG8KJZnvUDEyjyDh0m9AX0TPRo3xL2xAdG78lmNvFBzNjwSRqa8Wx0pPqx6tz0sAoG8e3z8PVVqBz4z3oM9oOm6vbngA73tquY8YAeyvbWtnr1ApZK9OiHkPATyGbxfKWQ9QVUcus67qL0ptTS9rpS0vRjTnb0ZEJc9SeRvPVBp+r0utwC82qSRu49lSj1HRZU9h2b/vAOMxb30+eo80f70u9cxh701RQq+uyKYvcJfBr2GKIQ916f1vXl7n7049By8USIXvgJKXT6rWSu94augvQFjPLvQPCG97j6qPb4vWTvzLQ29yszfvZrrBL53w4u9Dc4jPsHTUDxeb6Q9BSvAvWirDz5Ipda9WSeavUc4qjuFmrG9UD/hvSxvET3vQy49O+OYvdCqlT06n6Y9iBeevWjjQb2Wgki9XCrmvFLqqb0ueIi9ez+GvMStDj7RjT+9YT5YPIq+Db2xncq8+iX3vVh+MzzCZAi+jEKsPcVGsbzZvCq86+B/vN4lGb3qd6W7jHUmPtQCfz3SlEi94GYKvD1rID3dWaE8djiLPXk/tz0ADvg8Gj0zPdMK2j2hxKo8XlsgvW3wS73lzwG8Jkt4vQz7tj2kIgy98ferPI61SrzfSDg9EHPqPSzLfz1tU1c9MiCKPO7ljD0XHg09+Vf5vZsV+L3enWq9wUHUPOOrwr0OTgY+jkBnPTSHtD1FkGK99X0FvMrHqD2NMHS8oFGcvSd8lb2/KMu8SOIgvSNenD3HArS8Jw2zvSIQRr1dINs9VTEJPiYWmLxzJxa+q1ImPTscLzyVy807ZnJzO+Dx27yujYu9rCGfPQ2X9DyVGia9yRDLvTVC7T3+LYu9WWX3vOCyrz0Aa7q9tc7KO8EsID5vZko9vP2pPPEgRTuRgNo8VQa+PZjKMT7rezS9G0NUPbhFJ74A46s9C3lrvdBkBLxXOLi85zfnuv/ZCL46q+e7qSYsvLb3lz3GvFi7sFZIvha+P7zQHYM7sALvvdOJhD3/rf29b3mBPQP6kT0AEIG9eDeyvbn3tzq1B2W8N2ByPZsn1L2zicq8FsP+vX4Na73ie+29zEHyPV5vtL0tYN+9C3UfPSMUTTxqTJm9afJMtzuHCr3IDHQ9DUa1vF5bmLsyM5e8ir0NvVQwXT3y9Dq9vSkOvQ1upj2quPm92aoAvNLE3bwCGJg9KfH7PE3sFL51HX29JP4GPSgRYT0i9cC89yBZvX1Frr3t3gM+XMuTPFbPaL1oZK+9Dh2aOxq5hj2NVuI9IXGzvahsQrzzkkY9TtRGPKUAcj1gnq29KBS4vUtWb70KEzU9zn1UPFJugL0nyt68EVALvdQUML1hWeo8FIj2PU1NJz56rA29duY9PAjJxTy8Eic9r160PRGILb2I0pS8gxWfPM8W4ryW6M29hY8HvnpxxL3eHLc965++PAKnCLw0bwO9jYSEO12JXT1I4Zi8tIqTPacxwT1n9ay8kXpSPe0yVr3moM28IxQAvrmrMr0eJg69B9+8PXgzob1KFyo8OddovcZCQj3i7q09Bya6PR2OAD3MXgO+NVaDPA/SGry5edM9jjVwvB/SaryUsiO90w4BPh3CGj39CaU9LNtIvB1mX76M6N883tjnvTqZ7zxURHe8ekwxO3AijjwweZm9INa9PZ6KlL291wA8Ia2bPOVLnT38j1w92bPVvQ74oL3PDim9lsrPPP7n9j2GRJw9IoCovcaHxT2TaD+8utTfO3EF3rzpqJe90VpOPW1E6Tx9idW7RLqTvYnwsr0nJPI6bRYpPfBDzbw='))

ENV = os.environ.copy()
ENV["PYTHONUNBUFFERED"] = "1"
ENV["NUMBA_CACHE_DIR"] = str(BASE/"numba-clearvoice")
ENV["HF_HOME"] = str(BASE/"huggingface-cache")
ENV["SPEECHBRAIN_CACHE"] = str(BASE/"speechbrain-cache"/"spkrec-ecapa-voxceleb")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    ENV["HF_TOKEN"] = hf_token
    ENV["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("Hugging Face credentials configured from Kaggle secret HF_TOKEN.")
else:
    print("HF_TOKEN secret was not found; only public Hugging Face models will work.")
def checked(command, **kwargs):
    return subprocess.run(command, env=ENV, check=True, **kwargs)

if not Path(PYTHON).is_file():
    bootstrap = BASE/"virtualenv-bootstrap"
    checked([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
             "virtualenv>=20.26,<21"])
    bootstrap_env = ENV.copy(); bootstrap_env["PYTHONPATH"] = str(bootstrap)
    subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages",
                    "--no-download", str(VENV)], env=bootstrap_env, check=True)

# Install ClearVoice without its old strict NumPy/OpenCV pins. Kaggle's current
# Torch/WhisperX numerical stack remains untouched.
checked([PYTHON, "-m", "pip", "install", "--no-deps", "clearvoice==0.1.2"])
checked([PYTHON, "-m", "pip", "install", "gdown", "librosa==0.10.2.post1",
         "rotary-embedding-torch==0.8.3", "scenedetect==0.6.6",
         "python-speech-features==0.6", "yamlargparse", "torchinfo", "pydub"])
checked([PYTHON, "-m", "pip", "install",
         "speechbrain==1.1.1", "faster-whisper==1.2.1"])
checked([PYTHON, "-c", "import torch,soundfile,speechbrain,faster_whisper; "
         "from clearvoice import ClearVoice; "
         "print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None); "
         "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'"])


## Find the preserved full-video audio


In [ ]:
import soundfile as sf
input_root = Path("/kaggle/input")
candidates = []
for path in input_root.rglob("*.wav"):
    try:
        info = sf.info(path)
    except Exception:
        continue
    duration = info.frames / info.samplerate
    if 430 <= duration <= 450:
        candidates.append((0 if path.name == "full-video.wav" else 1, path, duration))
if not candidates:
    raise RuntimeError("Attach the stage-checkpoints dataset containing the 7:18 full-video.wav.")
_, AUDIO, duration = sorted(candidates)[0]
print("Using:", AUDIO)
print("Duration:", round(duration, 2), "seconds")


## Run the five-exchange test


In [ ]:
command = [PYTHON, "-B", str(WORK/"run_mossformer2_separation_experiment.py"),
           "--audio", str(AUDIO),
           "--labels", str(WORK/"labels.json"),
           "--voice-priors", str(WORK/"voice_embeddings.npy"),
           "--output-dir", str(OUTPUT),
           "--context", "3", "--device", "cuda",
           "--hf-home", ENV["HF_HOME"]]
log_path = OUTPUT/"run.log"
with log_path.open("w") as log:
    process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    recent = []
    for line in process.stdout:
        log.write(line); log.flush(); print(line, end="")
        recent = (recent + [line.rstrip()])[-40:]
    if process.wait() != 0:
        raise RuntimeError("MossFormer2 failed. Last output:\n" + "\n".join(recent))
print(json.dumps(json.loads((OUTPUT/"report.json").read_text()), indent=2)[:12000])


## Download the results


In [ ]:
with (OUTPUT/"runtime-packages.txt").open("w") as packages:
    checked([PYTHON, "-m", "pip", "freeze"], stdout=packages)
shutil.make_archive(str(BASE/"mossformer2-comparison-results"), "zip", OUTPUT)
from IPython.display import FileLink, display
display(FileLink(str(BASE/"mossformer2-comparison-results.zip")))
